# `mdpp` Example: DCCM and Hydrogen Bonds

This notebook demonstrates:

- residue-level correlated motion via `compute_dccm`
- hydrogen-bond dynamics via `compute_hbonds`

These analyses are complementary for interpreting collective structural behavior.
            


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from mplplots.utils import auto_ticks

from mdpp.analysis.hbond import compute_hbonds, format_hbond_triplets
from mdpp.analysis.metrics import compute_dccm
from mdpp.core.trajectory import align_trajectory, load_trajectory
from mdpp.plots.matrix import plot_dccm
from mdpp.plots.timeseries import plot_hbond_counts, plot_hbond_occupancy

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
)
traj = align_trajectory(traj, atom_selection="name CA")

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## DCCM


In [ ]:
dccm_result = compute_dccm(
    traj,
    atom_selection="name CA",
    align=False,
)

fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=120)
plot_dccm(dccm_result, ax=ax, add_colorbar=True)
ax.set_title("CA Dynamic Cross-Correlation Matrix")
auto_ticks(ax)
fig.tight_layout()

## Hydrogen Bonds


In [ ]:
hbond_result = compute_hbonds(
    traj,
    method="baker_hubbard",
    exclude_water=True,
    periodic=True,
    freq=0.1,
)
hbond_labels = format_hbond_triplets(traj.topology, hbond_result.triplets)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), dpi=120)
plot_hbond_counts(hbond_result, ax=axes[0])
axes[0].set_title("Hydrogen Bond Counts Over Time")

plot_hbond_occupancy(
    hbond_result,
    ax=axes[1],
    labels=hbond_labels,
    top_n=15,
)
axes[1].set_title("Top 15 Hydrogen Bond Occupancies")

auto_ticks(axes[0])
fig.tight_layout()

## Optional: Tabulate Highest-Occupancy H-Bonds


In [ ]:
summary = pd.DataFrame({
    "hbond": hbond_labels,
    "occupancy_percent": hbond_result.occupancy * 100.0,
}).sort_values("occupancy_percent", ascending=False)

summary.head(15)